# Практика: воспроизводимый ML-запуск через CLI

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

SCRIPT = Path("train_cli.py")
_DATA_URL = "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/modules/08_10_churn_logreg/data/bank_marketing_slim.csv"
DATA_PATH = next(
    (p for p in (
        Path("bank_marketing_slim.csv"),
        Path("../../data/bank_marketing_slim.csv"),
        Path("../data/bank_marketing_slim.csv"),
    ) if p.exists()),
    _DATA_URL,
)
assert SCRIPT.exists() and DATA_PATH.exists()


## 1. Команда как список аргументов

In [ ]:
cmd = None  # TODO: sys.executable, script, --data, path, --threshold, 0.45
assert isinstance(cmd, list) and len(cmd) == 6
assert cmd[0] == sys.executable
assert "--data" in cmd and "--threshold" in cmd


## 2. Запуск и диагностика

In [ ]:
proc = None  # TODO: subprocess.run(..., capture_output=True, text=True, check=False)
assert proc is not None and proc.returncode == 0, getattr(proc, "stderr", "")
assert proc.stdout.strip().startswith("{")


## 3. JSON-контракт

In [ ]:
metrics = None  # TODO: json.loads
expected_keys = {"threshold", "duration_in_features", "accuracy", "precision", "recall", "f1"}
assert set(metrics) == expected_keys
assert metrics["duration_in_features"] is False
assert all(0 <= metrics[name] <= 1 for name in ("accuracy", "precision", "recall", "f1"))


## 4. Функция запуска

In [ ]:
def run_experiment(threshold, seed=63):
    # TODO: вернуть parsed JSON; при ошибке поднять RuntimeError со stderr
    ...


trial = run_experiment(0.5)
assert trial["threshold"] == 0.5
assert trial["duration_in_features"] is False


## 5. Серия порогов

In [ ]:
thresholds = [0.25, 0.35, 0.45, 0.55, 0.65]
runs = None  # TODO: список результатов
assert len(runs) == len(thresholds)
assert [row["threshold"] for row in runs] == thresholds
assert all(row["duration_in_features"] is False for row in runs)


## 6. Таблица результатов

In [ ]:
import pandas as pd
run_table = None  # TODO
assert isinstance(run_table, pd.DataFrame) and len(run_table) == len(thresholds)
assert {"threshold", "precision", "recall", "f1"} <= set(run_table.columns)
assert run_table["recall"].is_monotonic_decreasing


## 7. Acceptance gate

In [ ]:
checks = {
    "process_ok": None,
    "json_contract": None,
    "duration_forbidden": None,
    "metrics_in_range": None,
    "deterministic": None,
}  # TODO
assert set(checks.values()) == {True}


## 8. Ошибочный запуск

Запустите CLI с несуществующим CSV и сохраните диагностическое сообщение.

In [ ]:
bad_cmd = [sys.executable, str(SCRIPT), "--data", "missing.csv", "--threshold", "0.5"]
bad_proc = None  # TODO
error_text = None  # TODO: stderr + stdout
assert bad_proc.returncode != 0
assert isinstance(error_text, str) and len(error_text) > 20


## 9. Самостоятельно: отчёт запуска

Подготовьте короткий отчёт для коллеги, который не видел ноутбук. Укажите точную роль параметра `threshold`, лучший F1 из выполненной серии, результат проверки `duration_in_features` и то, как система повела себя на намеренно неверном пути к данным. Отделите успешный результат модели от инженерного статуса процесса: корректные метрики не компенсируют ненулевой код завершения, а успешный процесс не компенсирует нарушение leakage-guard.

In [ ]:
CLI_REPORT = ""  # TODO: команда, лучший F1, leakage guard, ошибка; 240+ символов
assert len(CLI_REPORT) >= 240
assert "duration" in CLI_REPORT.lower()
assert "threshold" in CLI_REPORT.lower()
